# 00 — Replay simulated light curves as a sequential per-night alert stream

**Goal.** Take the DP2-DDF-matched light curves already produced in
`01_simulateDP2_SNinDDF/11_simSNandKNDDF_frSimpleDP2visits.ipynb` for several transient classes
(SNe Ia, SLSN proxy, Kilonovae) and turn them into what a broker such as Fink would actually
receive: a **sequence of single-epoch alerts per DIA object, ordered by MJD**, arriving one at a
time. This notebook does not do any classification yet — it only builds and validates the
`AlertStreamReplay` utility that notebooks `01_...` onward will consume to build the
prior/likelihood/posterior machinery.

**Strategy.**
1. Load the `target.data` (generated catalog) and `dset.data` (survey-matched light curves) parquet
   outputs for each available transient class.
2. Tag every object with a globally unique id (`"<class>_<local_index>"`) and a `class` column,
   then concatenate into a single combined observation table and a single combined metadata table.
3. Implement `AlertStreamReplay`, a small class that, given an object id, replays its observations
   one at a time in MJD order (`iter_alerts`), and can also replay the *whole* combined stream in
   global chronological order across all objects (`iter_global_stream`), which is closer to what a
   real nightly broker feed looks like.
4. Sanity-check the replay (per-object light-curve length, per-band cadence, time coverage) and
   save the combined tables for reuse.

**Conventions.** English-only code/comments; kernel `conda_py313`; outputs go to
`data_out_00_replay/` and `figs_out_00_replay/`; figures saved as PDF+PNG.

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS
- **Creation date:** 2026-08-15
- **Last update:** 2026-08-15
- **mac**: python kernel = conda_py313

## 1. Imports and configuration

In [ ]:
# Standard library
from pathlib import Path
from dataclasses import dataclass, field
from typing import Iterator, Optional

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 100
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------
NB_TAG = "00_replay"

DATA_OUT_DIR = Path(f"data_out_{NB_TAG}")
FIGS_OUT_DIR = Path(f"figs_out_{NB_TAG}")
DATA_OUT_DIR.mkdir(exist_ok=True)
FIGS_OUT_DIR.mkdir(exist_ok=True)

# Source of the class-labelled, DP2-DDF-matched light curves produced by notebook 11 of
# 01_simulateDP2_SNinDDF/. Each entry is (label, output-dir short tag) as used in that notebook's
# TRANSIENT_CONFIG (`cfg["short"]`) -- edit/extend this dict as more classes get a matched dataset
# (e.g. once SNe II are run through the same DP2-DDF matching pipeline).
SIM01_DIR = Path("../01_simulateDP2_SNinDDF")

CLASSES = {
    "snia": dict(label="SNe Ia", dir=SIM01_DIR / "data_out_11_simSNkDDF_snia"),
    "slsn": dict(label="SLSN (nugent-hyper proxy)", dir=SIM01_DIR / "data_out_11_simSNkDDF_slsn"),
    "kne": dict(label="Kilonovae", dir=SIM01_DIR / "data_out_11_simSNkDDF_kne"),
}

for short, cfg in CLASSES.items():
    print(f"{short:6s} -> {cfg['dir']}  (exists: {cfg['dir'].exists()})")

## 2. Load and combine the per-class catalogs and light curves

Each class contributes two tables:
- `meta_<short>` (from `target.data`, i.e. `<short>_generated_catalog.parquet`): one row per
  generated object, columns include at least `z`, `t0`, `ra`, `dec`, `magabs`, `magobs`,
  `template` (SNe Ia additionally have SALT2 `x1`/`c`; kilonovae additionally have a viewing
  angle `theta`).
- `obs_<short>` (from `dset.data`, i.e. `<short>_dp2ddf_lightcurves.parquet`): one row per
  simulated *detection/observation*, columns `mjd`, `band`, `skynoise`, `gain`, `zp`, `fieldid`,
  `flux`, `fluxerr`, indexed by `(local_index, index_obs)`.

Only objects that were actually matched to at least one DP2-DDF visit appear in `dset.data`
(i.e. objects with `Ndetected >= 1`), which is exactly what we want: a broker only ever sees
objects with at least one alert.

In [ ]:
def load_class(short, cfg):
    # Load one class's generated catalog + DP2-DDF-matched light curves, and tag both with a
    # globally unique object id so different classes' local integer indices never collide once
    # concatenated.
    meta = pd.read_parquet(cfg["dir"] / f"{short}_generated_catalog.parquet")
    obs = pd.read_parquet(cfg["dir"] / f"{short}_dp2ddf_lightcurves.parquet")

    # dset.data has a MultiIndex (local_index, index_obs); keep only the observations for
    # objects that are actually present in meta (defensive, should already be the case).
    obs = obs.reset_index().rename(columns={"index": "local_index"})

    meta = meta.copy()
    meta["local_index"] = meta.index
    meta["class"] = short
    meta["obj_id"] = short + "_" + meta["local_index"].astype(str)

    obs["class"] = short
    obs["obj_id"] = short + "_" + obs["local_index"].astype(str)

    return meta, obs


meta_frames, obs_frames = [], []
for short, cfg in CLASSES.items():
    meta_c, obs_c = load_class(short, cfg)
    print(
        f"{short:6s}: {len(meta_c):6d} generated, {obs_c['obj_id'].nunique():6d} detected "
        f"(>=1 pt), {len(obs_c):7d} observations"
    )
    meta_frames.append(meta_c)
    obs_frames.append(obs_c)

# Combined metadata: only keep objects that have at least one observation (broker never sees the rest).
meta_all = pd.concat(meta_frames, ignore_index=True)
obs_all = pd.concat(obs_frames, ignore_index=True)

detected_ids = obs_all["obj_id"].unique()
meta_all = meta_all[meta_all["obj_id"].isin(detected_ids)].set_index("obj_id")
obs_all = obs_all.set_index("obj_id")

print(
    f"\nCombined: {len(meta_all)} detected objects, {len(obs_all)} observations, "
    f"{meta_all['class'].nunique()} classes"
)
meta_all["class"].value_counts()

## 3. `AlertStreamReplay`: sequential per-object and global chronological replay

Two ways to consume the combined observation table, mirroring the two things the online
classifier of notebook `01_...` will need:

- **Per-object replay** (`iter_alerts`): for a chosen `obj_id`, yield its observations one at a
  time in MJD order, each time returning the *cumulative history so far* -- this is what feeds
  the sequential/online posterior update for a single DIA object.
- **Global chronological replay** (`iter_global_stream`): interleave *all* objects' observations
  in true MJD order, exactly as a broker ingesting nightly visits would see them -- useful later
  for testing the classifier under realistic multi-object, multi-night operating conditions
  (e.g. for the active-follow-up / bandit extension).

In [ ]:
@dataclass
class AlertStreamReplay:
    """Replay a combined skysurvey observation table as a sequence of single-epoch alerts.

    Parameters
    ----------
    obs : pd.DataFrame
        Combined observation table, indexed by `obj_id`, with at least columns
        `mjd`, `band`, `flux`, `fluxerr`, `class`.
    meta : pd.DataFrame
        Combined per-object metadata table, indexed by `obj_id`, with at least column `class`
        and the generative parameters (z, t0, ra, dec, ...).
    """

    obs: pd.DataFrame
    meta: pd.DataFrame

    def __post_init__(self):
        # Pre-sort once so every per-object slice is already MJD-ordered.
        self.obs = self.obs.sort_values("mjd")

    @property
    def object_ids(self) -> np.ndarray:
        return self.meta.index.to_numpy()

    def object_class(self, obj_id: str) -> str:
        return self.meta.loc[obj_id, "class"]

    def get_lightcurve(self, obj_id: str) -> pd.DataFrame:
        # Full, MJD-sorted light curve for one object (all epochs at once -- for plotting/QA,
        # not for the online classifier, which should only ever see `iter_alerts`' partial history).
        return self.obs.loc[[obj_id]].sort_values("mjd").reset_index(drop=True)

    def iter_alerts(self, obj_id: str) -> Iterator[pd.DataFrame]:
        """Yield the cumulative alert history for `obj_id`, one new epoch at a time.

        Each yielded DataFrame is the full history *up to and including* the new alert -- this
        is exactly the `data` argument a sequential Bayesian classifier should condition on at
        that point in time. The number of rows grows by one at each iteration.
        """
        lc = self.get_lightcurve(obj_id)
        for i in range(1, len(lc) + 1):
            yield lc.iloc[:i]

    def iter_global_stream(self) -> Iterator[tuple]:
        """Yield (mjd, obj_id, alert_row) for every observation in the combined table, in true
        chronological order across *all* objects -- mimics a real nightly broker feed where
        alerts from unrelated objects are interleaved.
        """
        ordered = self.obs.sort_values("mjd")
        for obj_id, row in zip(ordered.index, ordered.to_dict("records")):
            yield row["mjd"], obj_id, row


replay = AlertStreamReplay(obs=obs_all, meta=meta_all)
print(f"{len(replay.object_ids)} objects loaded into the replay stream.")

## 4. Sanity check: replay one example object per class

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 4), sharey=False)

for ax, short in zip(np.atleast_1d(axes), CLASSES):
    # Pick the object of this class with the most observations, for a clean illustration.
    class_ids = meta_all.index[meta_all["class"] == short]
    n_obs_per_obj = obs_all.loc[class_ids.intersection(obs_all.index.unique())].groupby(level=0).size()
    example_id = n_obs_per_obj.idxmax()

    # Replay it alert by alert, keeping only the final (full) history for this plot -- the point
    # here is just to check iter_alerts() reproduces the full light curve, not to animate it.
    history = None
    n_steps = 0
    for history in replay.iter_alerts(example_id):
        n_steps += 1
    assert n_steps == len(replay.get_lightcurve(example_id)), "iter_alerts() step count mismatch"

    for band, sub in history.groupby("band"):
        ax.errorbar(sub["mjd"], sub["flux"], yerr=sub["fluxerr"], fmt="o", ms=4, label=band)
    ax.set_title(f"{CLASSES[short]['label']}\n{example_id} ({n_steps} alerts)")
    ax.set_xlabel("MJD")
    ax.set_ylabel("flux")
    ax.legend(fontsize=7)

fig.suptitle("Example object per class, replayed alert-by-alert via iter_alerts()", y=1.03)
fig.tight_layout()
fig.savefig(FIGS_OUT_DIR / "example_replayed_lightcurves.pdf")
fig.savefig(FIGS_OUT_DIR / "example_replayed_lightcurves.png")

## 5. Diagnostics: alert-stream statistics per class

In [ ]:
n_alerts_per_obj = obs_all.groupby(level=0).size().rename("n_alerts")
diag = meta_all.join(n_alerts_per_obj)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for short, sub in diag.groupby("class"):
    axes[0].hist(
        sub["n_alerts"],
        bins=np.arange(1, sub["n_alerts"].max() + 2) - 0.5,
        histtype="step",
        lw=1.5,
        label=CLASSES[short]["label"],
    )
axes[0].set_xlabel("number of alerts per object")
axes[0].set_ylabel("N objects")
axes[0].set_yscale("log")
axes[0].legend(fontsize=8)
axes[0].set_title("Alert-stream length distribution")

# Time coverage (last alert MJD - first alert MJD) per object -- how long a given classification
# problem stays "open" before the light curve stops being observed.
span = obs_all.groupby(level=0)["mjd"].agg(lambda s: s.max() - s.min()).rename("span_days")
diag = diag.join(span)
for short, sub in diag.groupby("class"):
    axes[1].hist(sub["span_days"], bins=30, histtype="step", lw=1.5, label=CLASSES[short]["label"])
axes[1].set_xlabel("time span covered (days)")
axes[1].set_ylabel("N objects")
axes[1].set_yscale("log")
axes[1].legend(fontsize=8)
axes[1].set_title("Observed time-coverage per object")

fig.tight_layout()
fig.savefig(FIGS_OUT_DIR / "alert_stream_diagnostics.pdf")
fig.savefig(FIGS_OUT_DIR / "alert_stream_diagnostics.png")

diag.groupby("class")[["n_alerts", "span_days"]].describe()

## 6. Save combined tables for reuse

`obs_all` and `meta_all` are the two tables notebook `01_...` (single-epoch likelihood) and
`02_...` (sequential model comparison) will load directly, instead of re-reading and re-tagging
the per-class parquet files from `01_simulateDP2_SNinDDF/`.

In [ ]:
obs_all.to_parquet(DATA_OUT_DIR / "combined_alert_stream_obs.parquet")
meta_all.to_parquet(DATA_OUT_DIR / "combined_alert_stream_meta.parquet")

print("Saved:")
for f in sorted(DATA_OUT_DIR.glob("*.parquet")):
    print(f"  - {f}")

## 7. Summary / next steps

- `AlertStreamReplay.iter_alerts(obj_id)` yields the growing per-object history exactly as an
  online classifier should consume it: never more than what has been observed so far.
- `AlertStreamReplay.iter_global_stream()` gives the fully interleaved, multi-object chronological
  feed, for later use in an active-follow-up / bandit extension.
- Combined, class-tagged tables are saved under `data_out_00_replay/` for direct reuse.

**Next (notebook `01_...`).** Implement `TransientModelWrapper.log_likelihood(theta, data)` for a
single skysurvey model (start with SNe Ia / SALT2, since its `x1`/`c`/`t0`/`z` parametrization is
the most standard), evaluate it against a single `iter_alerts()` snapshot, and check that the
likelihood is maximized near each object's true generative parameters (available in `meta_all`)
-- this validates the flux model + noise model before moving on to the multi-class evidence
comparison in notebook `02_...`.